In [ ]:
# Gentrification Multimodal GCN Classifier with 2031 Prediction, Heatmap, and BERTopic Explanation

import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import geopandas as gpd
from libpysal.weights import Queen
from shapely.geometry import Polygon
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# === 1. Load and prepare data ===
# Assumes df_train (2002–2011 data) and df_future (2012–2021) are available
# gdf (GeoDataFrame) contains 2021 LSOA geometries for visualization

# --- 1.1 Construct edge_index and edge_weight from shapefile ---
gdf = gpd.read_file("LSOA_2021.shp").sort_values("LSOA21CD").reset_index(drop=True)
w = Queen.from_dataframe(gdf)
edges, weights = [], []
for src, neighbors in w.neighbors.items():
    for tgt in neighbors:
        geom_src = gdf.geometry.iloc[src]
        geom_tgt = gdf.geometry.iloc[tgt]
        inter = geom_src.intersection(geom_tgt)
        weight = inter.length if inter.length > 0 else 0.001
        edges.append([src, tgt])
        weights.append(weight)
        edges.append([tgt, src])
        weights.append(weight)
edge_index = torch.tensor(edges, dtype=torch.long).T
edge_weight = torch.tensor(weights, dtype=torch.float32)

# --- 1.2 BERTopic topic extraction ---
def extract_topic_features(df):
    texts = df['planning_text'].fillna("").tolist()
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
    topics, probs = topic_model.transform(texts, embeddings)
    return torch.tensor(np.nan_to_num(probs), dtype=torch.float32)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(verbose=False)

# Train BERTopic on historical text
topics_train, probs_train = topic_model.fit_transform(
    df_train['planning_text'].fillna("").tolist(),
    embedding_model.encode(df_train['planning_text'].fillna("").tolist(), show_progress_bar=True)
)
topic_features_train = torch.tensor(np.nan_to_num(probs_train), dtype=torch.float32)

# Future text embedding
topic_features_future = extract_topic_features(df_future)

# --- 1.3 Tabular Features ---
tabular_cols = ['imd_decile', 'pct_owner_occupied', 'pct_professional']
scaler = StandardScaler()
tabular_train = torch.tensor(scaler.fit_transform(df_train[tabular_cols]), dtype=torch.float32)
tabular_future = torch.tensor(scaler.transform(df_future[tabular_cols]), dtype=torch.float32)

# --- 1.4 Combine features ---
x_train = torch.cat([topic_features_train, tabular_train], dim=1)
x_future = torch.cat([topic_features_future, tabular_future], dim=1)
y_train = torch.tensor(df_train['gentrification_class'].values, dtype=torch.long)

# --- 1.5 Build PyG Data ---
data = Data(x=x_train, edge_index=edge_index, edge_weight=edge_weight, y=y_train)

# === 2. Train/Val Split ===
indices = np.arange(data.num_nodes)
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=y_train.numpy())
train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
data.train_mask = train_mask
data.val_mask = val_mask

# === 3. Model ===
class GentrificationGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, num_classes)
        )

    def forward(self, x, edge_index, edge_weight):
        x = self.gcn1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.gcn2(x, edge_index, edge_weight)
        return self.classifier(x)

# === 4. Training ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)
model = GentrificationGCN(x_train.shape[1], 256, 5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.edge_weight)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# === 5. Evaluation ===
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_weight)
    preds_val = logits[data.val_mask].argmax(dim=1)
    acc = (preds_val == data.y[data.val_mask]).float().mean()
    print(f"Validation Accuracy: {acc:.4f}")

# === 6. Predict 2031 classes ===
x_future = x_future.to(device)
preds_future = model(x_future, edge_index.to(device), edge_weight.to(device)).argmax(dim=1).cpu().numpy()
df_future['gentrification_class_pred_2031'] = preds_future

# === 7. Plot Heatmap of Predicted Classes ===
gdf["class2031"] = preds_future
gdf.plot(column="class2031", cmap="RdYlBu_r", legend=True, figsize=(10, 10))
plt.title("Predicted Gentrification Classes for 2031")
plt.axis("off")
plt.tight_layout()
plt.show()

# === 8. BERTopic Semantic Explanation per Class ===
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances

topic_distributions = topic_features_future.numpy()
preds = df_future['gentrification_class_pred_2031']
mean_topic_by_class = pd.DataFrame(topic_distributions).groupby(preds).mean()

# Display top 3 topics for each class
for c in range(5):
    top_topics = mean_topic_by_class.loc[c].sort_values(ascending=False).head(3).index
    print(f"\nClass {c} top BERTopic topics:")
    for topic_id in top_topics:
        print(f"Topic {topic_id}: {topic_model.get_topic(topic_id)}")
